<a href="https://colab.research.google.com/github/GAIA-HKUSTGZ/UCUG1002_AI-Society/blob/main/labs/2_Network/lab_b_mini_think_on_graph_reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab B: Mini Think-on-Graph Reasoning

**Course:** AI + Society  
**Topic:** LLM-style path reasoning over a knowledge graph  
**Estimated time:** 60-75 minutes

This notebook implements a small classroom version of Think-on-Graph (ToG). It is not the full research system. The goal is to make the mechanism visible:

1. Link the question to starting entities.
2. Retrieve neighboring triples.
3. Score which relations look promising.
4. Expand the graph frontier with beam search.
5. Return an evidence path that students can inspect.

The notebook is **LLM-first with an offline fallback**. If an LLM API key is available, the LLM scores candidate graph steps during frontier expansion. If no API key is available, or if `USE_LLM_REASONING = False`, the same graph search runs with transparent keyword scoring.

## How to open this lab in Google Colab

Recommended classroom path:

1. Open the Colab link shared by your instructor.
2. Click **File -> Save a copy in Drive** to create your own editable copy.
3. Add your API key in Colab **Secrets** if you want to use a real LLM.
4. Run cells from top to bottom, or use **Runtime -> Run all**.

Fallback path if your instructor gives you only the `.ipynb` file:

1. Go to Google Colab: https://colab.research.google.com/
2. Choose **Upload**.
3. Click **Choose file**, then select this notebook file.
4. After it opens, click **File -> Save a copy in Drive**.

If you want to use a real LLM, add your API key in Colab **Secrets** before running the LLM cells.

## Student roadmap

This lab has one central idea: instead of retrieving a paragraph, an LLM-guided graph reasoner can move from entity to entity and build an evidence path.

| Part | What you run | What you should understand | What to observe |
|---|---|---|---|
| Setup | Install/import packages and configure LLM provider | The lab can use Gemini/OpenAI, but it can also fall back to transparent keyword scoring | The mode label tells you whether LLM or offline scoring was used |
| Knowledge graph | Build triples such as `A -[relation]-> B` | A graph stores small pieces of relational evidence | Each edge carries an interpretable reason |
| Entity linking | Match the question to starting nodes | Graph reasoning must decide where to begin | Bad starting entities can lead to bad paths |
| Frontier expansion | Explore neighboring triples | Reasoning becomes a search problem over graph edges | Beam width and depth control exploration |
| LLM scoring | Ask the LLM to score candidate graph steps | The LLM acts as a controller, not just an answer generator | The path remains visible even when the LLM guides it |
| Answer from paths | Convert evidence paths into a short answer | The answer should be grounded in traceable graph evidence | A path is useful evidence, not automatic proof |
| Compare retrieval | Compare graph paths with top-k text snippets | Retrieval finds relevant text; graph reasoning composes relations | Some questions need multi-hop structure |

In [ ]:
import importlib
import subprocess
import sys

required = {
    "networkx": "networkx",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "openai": "openai",
    "requests": "requests",
}

for module, package in required.items():
    try:
        importlib.import_module(module)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import re
import math
import os
import json
import getpass
import textwrap
from collections import defaultdict
from difflib import SequenceMatcher

import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_colwidth", 100)
plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Setup complete.")

## 0. API key setup

This Mini-ToG lab tries to use a real LLM controller by default, but it still runs without a key.

### Get a Gemini API key

1. Visit Google AI Studio: https://aistudio.google.com/
2. Sign in with your Google account.
3. Click **Get API key** in the left sidebar, then click **Create API key**.
4. Select a Google Cloud project. If you do not have one, Google AI Studio can create a default project for you.
5. Copy the API key.

### Add the key to Colab

1. In Colab, open **Secrets** in the left sidebar.
2. Add a new secret named `GEMINI_API_KEY`.
3. Paste your copied key as the value.
4. Keep notebook access enabled for this secret.

OpenAI is also supported: add `OPENAI_API_KEY` instead of `GEMINI_API_KEY`.

If you are not using Colab Secrets, set `PROMPT_FOR_API_KEY = True` and paste the key into the hidden password prompt.

Never hard-code a key in a notebook cell.

In [ ]:
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "auto").lower()  # "auto", "gemini", or "openai"
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
USE_LLM_REASONING = True
PROMPT_FOR_API_KEY = False  # Keep False for automatic fallback; set True to ask for a hidden key prompt.


def _load_colab_secret(secret_name):
    try:
        from google.colab import userdata

        return userdata.get(secret_name)
    except Exception:
        return None


def try_load_provider_key(provider, prompt_if_missing=False):
    provider = provider.lower()
    env_name = "GEMINI_API_KEY" if provider == "gemini" else "OPENAI_API_KEY"
    if os.environ.get(env_name):
        return True

    key = _load_colab_secret(env_name)
    if key:
        os.environ[env_name] = key
        return True

    if not prompt_if_missing:
        return False

    label = "Gemini" if provider == "gemini" else "OpenAI"
    key = getpass.getpass(f"Paste your {label} API key. It will not be printed: ").strip()
    if not key:
        return False
    os.environ[env_name] = key
    return True


def resolve_llm_provider(prompt_if_missing=False):
    if LLM_PROVIDER in {"gemini", "google"}:
        return "gemini" if try_load_provider_key("gemini", prompt_if_missing) else None
    if LLM_PROVIDER == "openai":
        return "openai" if try_load_provider_key("openai", prompt_if_missing) else None

    if try_load_provider_key("gemini", False):
        return "gemini"
    if try_load_provider_key("openai", False):
        return "openai"
    if prompt_if_missing:
        return "gemini" if try_load_provider_key("gemini", True) else None
    return None


def ensure_llm_api_key():
    provider = resolve_llm_provider(prompt_if_missing=True)
    if not provider:
        raise ValueError("No Gemini or OpenAI API key was provided.")
    print(f"{provider} API key is available for this runtime.")
    return provider


def llm_text(prompt, temperature=0):
    provider = ensure_llm_api_key()
    if provider == "gemini":
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"
        headers = {
            "x-goog-api-key": os.environ["GEMINI_API_KEY"],
            "Content-Type": "application/json",
        }
        body = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": temperature, "maxOutputTokens": 1024},
        }
        response = requests.post(url, headers=headers, json=body, timeout=60)
        response.raise_for_status()
        data = response.json()
        parts = data["candidates"][0]["content"].get("parts", [])
        return "".join(part.get("text", "") for part in parts).strip()

    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(model=OPENAI_MODEL, input=prompt, temperature=temperature)
    return response.output_text.strip()


print(f"LLM_PROVIDER = {LLM_PROVIDER}")
print(f"Gemini model: {GEMINI_MODEL}; OpenAI model: {OPENAI_MODEL}")
print(f"USE_LLM_REASONING = {USE_LLM_REASONING}")
print("If no API key is found, the notebook falls back to keyword graph reasoning.")

## 1. Build a small knowledge graph

Each row below is a triple:

```text
source --relation--> target
```

The graph describes disaster response, misinformation, evacuation pressure, public agencies, and AI tools.

In [ ]:
NODE_META = {
    "Hurricane Iris": {"type": "Event", "description": "major storm approaching the coast"},
    "Bridge Flooding": {"type": "Event", "description": "infrastructure disruption in a high-risk area"},
    "Rumor Surge": {"type": "Event", "description": "misleading claims spread through social media"},
    "Harbor District": {"type": "Region", "description": "coastal district with high flood risk"},
    "River County": {"type": "Region", "description": "riverine region with medium-high flood risk"},
    "Inland Campus": {"type": "Region", "description": "low-risk area that can still create shadow evacuation"},
    "High Flood Risk": {"type": "Risk", "description": "high exposure to flood hazard"},
    "Low Flood Risk": {"type": "Risk", "description": "low exposure to flood hazard"},
    "Shadow Evacuation": {"type": "Behavior", "description": "low-risk residents leave early and occupy road capacity"},
    "Harbor Evacuation": {"type": "Behavior", "description": "high-risk evacuation from Harbor District"},
    "Emergency Agency": {"type": "Organization", "description": "government emergency response agency"},
    "Transit Authority": {"type": "Organization", "description": "public transportation operator"},
    "University Lab": {"type": "Organization", "description": "research group building AI tools"},
    "Community Volunteers": {"type": "Organization", "description": "civil society group helping residents"},
    "Evacuation Order": {"type": "Policy", "description": "emergency policy ordering high-risk evacuation"},
    "Road Closure Plan": {"type": "Policy", "description": "transport policy to keep key routes usable"},
    "Misinformation Response": {"type": "Policy", "description": "policy for countering rumor spread"},
    "AI Damage Assessment": {"type": "Policy", "description": "policy using AI to assess infrastructure damage"},
    "Graph Alert Agent": {"type": "Tool", "description": "AI agent that reasons over a graph of events and actors"},
    "Damage Classifier": {"type": "Tool", "description": "computer vision model for detecting flood damage"},
    "Local Influencer": {"type": "Person", "description": "media actor with many followers"},
    "Resident Group": {"type": "Person", "description": "citizen group affected by information diffusion"},
    "Mayor Chen": {"type": "Person", "description": "public official coordinating response"},
    "Prof. Rivera": {"type": "Person", "description": "scientist advising emergency analytics"},
    "Hospital": {"type": "Place", "description": "critical facility that needs accessible routes"},
}

TRIPLES = [
    ("Hurricane Iris", "caused", "Bridge Flooding", "The storm caused flooding near a key bridge."),
    ("Hurricane Iris", "affected", "River County", "The storm track crossed River County."),
    ("Bridge Flooding", "occurred_in", "Harbor District", "The bridge disruption happened in Harbor District."),
    ("Harbor District", "has_risk", "High Flood Risk", "The area has high flood exposure."),
    ("River County", "has_risk", "High Flood Risk", "The county has riverine flood exposure."),
    ("Inland Campus", "has_risk", "Low Flood Risk", "The campus is away from the flood zone."),
    ("Inland Campus", "can_create", "Shadow Evacuation", "Low-risk residents can still leave early."),
    ("Shadow Evacuation", "competes_for_road_capacity_with", "Harbor Evacuation", "Early low-risk departures consume road capacity."),
    ("Harbor Evacuation", "protects", "Harbor District", "High-risk residents need road capacity first."),
    ("Emergency Agency", "issued", "Evacuation Order", "The agency issued the evacuation order."),
    ("Evacuation Order", "targets", "Harbor District", "The order focuses on the high-risk district."),
    ("Transit Authority", "implements", "Road Closure Plan", "Transit officials implement closures and priority routes."),
    ("Road Closure Plan", "protects_route_to", "Hospital", "The plan keeps access to the hospital open."),
    ("Hospital", "located_in", "River County", "The hospital serves River County."),
    ("University Lab", "built", "Graph Alert Agent", "The university lab built the graph alert agent."),
    ("Graph Alert Agent", "supports", "Emergency Agency", "The agent supports emergency coordination."),
    ("University Lab", "shares_data_with", "Emergency Agency", "The lab shares risk and event data with the agency."),
    ("Emergency Agency", "implements", "Misinformation Response", "The agency runs the misinformation response policy."),
    ("Misinformation Response", "counters", "Rumor Surge", "The policy counters the rumor surge."),
    ("Rumor Surge", "amplified_by", "Local Influencer", "The influencer amplified misleading claims."),
    ("Local Influencer", "influences", "Resident Group", "Followers include local residents."),
    ("Resident Group", "located_in", "Inland Campus", "The group is mainly in the low-risk campus area."),
    ("AI Damage Assessment", "uses_tool", "Damage Classifier", "Damage assessment uses a computer vision model."),
    ("Damage Classifier", "detects", "Bridge Flooding", "The classifier detects flooded bridge imagery."),
    ("Damage Classifier", "supports", "Emergency Agency", "The classifier supports emergency response with visual damage evidence."),
    ("Mayor Chen", "coordinates_with", "Emergency Agency", "The mayor coordinates public response with the agency."),
    ("Prof. Rivera", "advises", "University Lab", "The professor advises the lab team."),
]

G = nx.MultiDiGraph()
for node, meta in NODE_META.items():
    G.add_node(node, **meta)

for source, relation, target, evidence in TRIPLES:
    G.add_edge(source, target, relation=relation, evidence=evidence)

triples_df = pd.DataFrame(TRIPLES, columns=["source", "relation", "target", "evidence"])
triples_df.head(12)

In [ ]:
def draw_graph(graph, highlight_nodes=None):
    highlight_nodes = set(highlight_nodes or [])
    pos = nx.spring_layout(graph, seed=9, k=0.9)
    node_colors = []
    for node in graph.nodes:
        node_colors.append("#f4b183" if node in highlight_nodes else "#d9eaf7")

    plt.figure(figsize=(12, 8))
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, edgecolors="#2d4f73", node_size=1150)
    nx.draw_networkx_labels(graph, pos, font_size=8)
    nx.draw_networkx_edges(graph, pos, edge_color="#8b9aaa", arrows=True, arrowsize=14, width=1.2)

    edge_labels = {}
    for u, v, key, data in graph.edges(keys=True, data=True):
        edge_labels[(u, v)] = data["relation"]
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=7)

    plt.title("Mini AI + Society Knowledge Graph")
    plt.axis("off")
    plt.show()


draw_graph(G)

## 2. Link the question to starting entities

The full ToG system uses an LLM and a large KG. Here we use transparent string matching so students can inspect every step.

In [ ]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "by", "to", "of", "and", "or",
    "for", "in", "on", "with", "from", "how", "what", "which", "who", "why", "can",
    "does", "do", "did", "this", "that", "through", "connected", "connects", "connection",
}


def tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", text.lower().replace("_", " "))
    return [tok for tok in tokens if tok not in STOPWORDS and len(tok) > 1]


def node_text(node):
    meta = G.nodes[node]
    return f"{node} {meta.get('type', '')} {meta.get('description', '')}"


def link_entities(question, top_k=4):
    q_tokens = set(tokenize(question))
    rows = []
    for node in G.nodes:
        n_tokens = set(tokenize(node_text(node)))
        overlap = len(q_tokens & n_tokens)
        string_sim = SequenceMatcher(None, question.lower(), node.lower()).ratio()
        phrase_bonus = 2.0 if node.lower() in question.lower() else 0.0
        score = overlap + phrase_bonus + string_sim
        if score > 0:
            rows.append({"entity": node, "type": G.nodes[node]["type"], "score": round(score, 3)})
    rows = sorted(rows, key=lambda row: row["score"], reverse=True)[:top_k]
    return rows


question = "How is the Local Influencer connected to misinformation response?"
pd.DataFrame(link_entities(question))

## 3. Expand the graph frontier

At each step, the algorithm looks at neighboring triples and asks:

> Which next relation looks most useful for answering the question?

In the original ToG idea, this scoring can be done by an LLM. Here we use keyword overlap so the mechanism stays visible.

In [ ]:
def step_score(question, relation, next_node, evidence):
    q_tokens = set(tokenize(question))
    # Score only the next relation and node. Evidence text is shown later,
    # but not used here, so the search does not get rewarded for walking
    # away from the question because an evidence sentence repeats old terms.
    candidate_text = f"{relation} {next_node} {node_text(next_node)}"
    c_tokens = set(tokenize(candidate_text))
    overlap = len(q_tokens & c_tokens)
    relation_bonus = 0.25 if any(tok in relation.lower() for tok in q_tokens) else 0.0
    return overlap + relation_bonus + 0.05


def neighbor_steps(graph, current_node):
    # Outgoing edges keep their natural direction.
    for _, target, key, data in graph.out_edges(current_node, keys=True, data=True):
        yield {
            "next_node": target,
            "source": current_node,
            "relation": data["relation"],
            "target": target,
            "direction": "out",
            "evidence": data["evidence"],
        }

    # Incoming edges are also useful for reasoning: A -> current can tell us who caused or supports current.
    for source, _, key, data in graph.in_edges(current_node, keys=True, data=True):
        yield {
            "next_node": source,
            "source": source,
            "relation": data["relation"],
            "target": current_node,
            "direction": "in",
            "evidence": data["evidence"],
        }


def format_step(step):
    return f"{step['source']} -[{step['relation']}]-> {step['target']}"


def format_path(path):
    if not path["steps"]:
        return path["nodes"][0]
    return " | ".join(format_step(step) for step in path["steps"])


def mini_think_on_graph(question, max_depth=3, beam_width=5, start_top_k=3):
    starts = link_entities(question, top_k=start_top_k)
    if not starts:
        return [], []

    linked_entities = [row["entity"] for row in starts]
    frontier = [
        {"nodes": [row["entity"]], "steps": [], "score": row["score"], "start": row["entity"]}
        for row in starts
    ]
    trace = []
    all_paths = []

    for depth in range(1, max_depth + 1):
        candidates = []
        for path in frontier:
            current = path["nodes"][-1]
            visited = set(path["nodes"])
            for step in neighbor_steps(G, current):
                if step["next_node"] in visited:
                    continue
                s = step_score(question, step["relation"], step["next_node"], step["evidence"])
                new_path = {
                    "nodes": path["nodes"] + [step["next_node"]],
                    "steps": path["steps"] + [step],
                    "score": path["score"] + s - 0.10 * (depth - 1),
                    "start": path["start"],
                }
                candidates.append(new_path)
                all_paths.append(new_path)
                trace.append(
                    {
                        "depth": depth,
                        "from": current,
                        "candidate_step": format_step(step),
                        "step_score": round(s, 3),
                        "path_score": round(new_path["score"], 3),
                    }
                )

        frontier = sorted(candidates, key=lambda p: p["score"], reverse=True)[:beam_width]
        if not frontier:
            break

    def rank_score(path):
        path_tokens = set(tokenize(" ".join(path["nodes"] + [step["relation"] for step in path["steps"]])))
        q_tokens = set(tokenize(question))
        coverage = len(q_tokens & path_tokens)
        linked_hits = sum(1 for entity in linked_entities if entity in path["nodes"])
        primary_hits = sum(1 for entity in linked_entities[:2] if entity in path["nodes"])
        length_penalty = 0.35 * len(path["steps"])
        path["rank_score"] = path["score"] + 0.60 * linked_hits + 1.50 * primary_hits + 0.25 * coverage - length_penalty
        return path["rank_score"]

    ranked_paths = sorted(all_paths, key=rank_score, reverse=True)
    return ranked_paths, trace

## 4. LLM-first graph reasoning

The wrapper below tries to use a real LLM as the graph controller. At each frontier step, the LLM scores candidate triples. If no API key is available, or if `USE_LLM_REASONING = False`, it falls back to the keyword scorer above.

In [ ]:
def llm_score_candidate_steps(question, current_node, candidate_steps):
    items = []
    for i, step in enumerate(candidate_steps):
        items.append(
            {
                "id": i,
                "triple": format_step(step),
                "next_node_type": G.nodes[step["next_node"]]["type"],
                "next_node_description": G.nodes[step["next_node"]]["description"],
                "evidence": step["evidence"],
            }
        )

    prompt = f'''
You are the graph-navigation controller for a Mini Think-on-Graph lab.

Question: {question}
Current node: {current_node}

Candidate graph steps:
{json.dumps(items, indent=2)}

Return only valid JSON in this form:
[
  {{"id": 0, "score": 0.0, "reason": "short reason"}},
  {{"id": 1, "score": 0.0, "reason": "short reason"}}
]

Use scores from 0 to 5. A high score means this graph step is useful for answering the question.
'''

    raw = llm_text(prompt, temperature=0)
    try:
        ratings = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"```(?:json)?\s*(.*?)```", raw, re.DOTALL)
        if not match:
            raise
        ratings = json.loads(match.group(1))

    by_id = {int(row["id"]): float(row["score"]) for row in ratings}
    return [by_id.get(i, 0.0) for i in range(len(candidate_steps))]


def mini_think_on_graph_with_llm(question, max_depth=2, beam_width=3, start_top_k=2):
    starts = link_entities(question, top_k=start_top_k)
    if not starts:
        return [], []

    frontier = [
        {"nodes": [row["entity"]], "steps": [], "score": row["score"], "start": row["entity"]}
        for row in starts
    ]
    trace = []
    all_paths = []

    for depth in range(1, max_depth + 1):
        candidates = []
        for path in frontier:
            current = path["nodes"][-1]
            steps = [step for step in neighbor_steps(G, current) if step["next_node"] not in set(path["nodes"])]
            if not steps:
                continue
            llm_scores = llm_score_candidate_steps(question, current, steps)
            for step, s in zip(steps, llm_scores):
                new_path = {
                    "nodes": path["nodes"] + [step["next_node"]],
                    "steps": path["steps"] + [step],
                    "score": path["score"] + s,
                    "start": path["start"],
                }
                candidates.append(new_path)
                all_paths.append(new_path)
                trace.append(
                    {
                        "depth": depth,
                        "from": current,
                        "candidate_step": format_step(step),
                        "llm_step_score": round(s, 3),
                        "path_score": round(new_path["score"], 3),
                    }
                )
        frontier = sorted(candidates, key=lambda p: p["score"], reverse=True)[:beam_width]
        if not frontier:
            break

    return sorted(all_paths, key=lambda p: p["score"], reverse=True), trace


def run_graph_reasoning(question, use_llm=USE_LLM_REASONING, prompt_for_key=PROMPT_FOR_API_KEY):
    use_real_llm = use_llm and resolve_llm_provider(prompt_if_missing=prompt_for_key)
    if use_real_llm:
        try:
            paths, trace = mini_think_on_graph_with_llm(question, max_depth=2, beam_width=3, start_top_k=2)
            return paths, trace, "llm_guided_graph_reasoning"
        except Exception as exc:
            print(f"LLM graph reasoning failed or was unavailable ({exc}). Falling back to keyword graph reasoning.")

    paths, trace = mini_think_on_graph(question, max_depth=3, beam_width=5, start_top_k=3)
    return paths, trace, "offline_keyword_graph_reasoning"

In [ ]:
demo_question = "Which organization connects the University Lab to the evacuation order?"
paths, trace, reasoning_mode = run_graph_reasoning(demo_question)

print("Question:", demo_question)
print("Mode:", reasoning_mode)
print("\nEntity linking:")
display(pd.DataFrame(link_entities(demo_question)))
print("\nTop reasoning paths:")
display(pd.DataFrame(
    [{"rank": i + 1, "score": round(path["score"], 3), "path": format_path(path)} for i, path in enumerate(paths[:5])]
))
print("\nFrontier trace, first 10 candidates:")
display(pd.DataFrame(trace).head(10))

## 5. Turn paths into an answer

The answer should be traceable. A graph-reasoning system is useful in social applications only if we can see the evidence path, not just the final sentence.

In [ ]:
def answer_from_paths(question, paths, top_n=2):
    if not paths:
        return "No plausible reasoning path was found."
    best = paths[0]
    evidence_lines = [format_path(path) for path in paths[:top_n]]
    answer = [
        f"Question: {question}",
        "",
        "A graph-based answer is supported by the following evidence path:",
        evidence_lines[0],
    ]
    if len(evidence_lines) > 1:
        answer.append("")
        answer.append("Additional path to inspect:")
        answer.append(evidence_lines[1])
    answer.append("")
    answer.append("Interpretation: the path suggests a connected chain of evidence. It is not proof by itself; it is a trace for inspection.")
    return "\n".join(answer)


print(answer_from_paths(demo_question, paths))

In [ ]:
test_questions = [
    "How is the Local Influencer connected to misinformation response?",
    "Which AI tool helps the Emergency Agency understand Bridge Flooding?",
    "Why can delaying low-risk evacuation reduce pressure on Harbor District?",
    "Which organization connects the University Lab to the evacuation order?",
]

for q in test_questions:
    paths, _, reasoning_mode = run_graph_reasoning(q)
    print("=" * 100)
    print("Mode:", reasoning_mode)
    print(answer_from_paths(q, paths, top_n=1))

## 6. Visualize the top path

Visualization helps students understand why graph reasoning feels different from paragraph retrieval.

In [ ]:
def draw_path(graph, path):
    sub_nodes = set(path["nodes"])
    sub_edges = [(step["source"], step["target"]) for step in path["steps"]]
    pos = nx.spring_layout(graph.subgraph(sub_nodes), seed=11)

    plt.figure(figsize=(9, 5))
    nx.draw_networkx_nodes(graph.subgraph(sub_nodes), pos, node_color="#fee8c8", edgecolors="#7f4f24", node_size=1600)
    nx.draw_networkx_labels(graph.subgraph(sub_nodes), pos, font_size=9)
    nx.draw_networkx_edges(graph.subgraph(sub_nodes), pos, edgelist=sub_edges, edge_color="#0b376d", arrows=True, arrowsize=18, width=2.5)

    labels = {}
    for step in path["steps"]:
        labels[(step["source"], step["target"])] = step["relation"]
    nx.draw_networkx_edge_labels(graph.subgraph(sub_nodes), pos, edge_labels=labels, font_size=8)
    plt.title("Top Reasoning Path")
    plt.axis("off")
    plt.show()


if paths:
    draw_path(G, paths[0])

## 7. Compare with simple top-k text retrieval

Top-k retrieval finds individually relevant snippets. Graph reasoning tries to compose multiple edges into a path.

In [ ]:
DOCUMENTS = [
    {
        "text": f"{source} {relation.replace('_', ' ')} {target}. {evidence}",
        "source": source,
        "relation": relation,
        "target": target,
    }
    for source, relation, target, evidence in TRIPLES
]


def retrieve_chunks(question, top_k=5):
    q_tokens = set(tokenize(question))
    rows = []
    for doc in DOCUMENTS:
        d_tokens = set(tokenize(doc["text"]))
        score = len(q_tokens & d_tokens)
        if score > 0:
            rows.append({"score": score, "text": doc["text"]})
    return sorted(rows, key=lambda row: row["score"], reverse=True)[:top_k]


question = "Which organization connects the University Lab to the evacuation order?"
print("Top-k text retrieval:")
display(pd.DataFrame(retrieve_chunks(question)))

print("\nGraph reasoning:")
paths, _, reasoning_mode = run_graph_reasoning(question)
print("Mode:", reasoning_mode)
display(pd.DataFrame(
    [{"rank": i + 1, "score": round(path["score"], 3), "path": format_path(path)} for i, path in enumerate(paths[:3])]
))

## 8. Student tasks

1. Run once with `USE_LLM_REASONING = True` and a valid key. Then run once with `USE_LLM_REASONING = False`. Does the top path change?

2. In offline mode, increase `beam_width` from 5 to 10. Does the top path change?

3. In offline mode, increase `max_depth` from 3 to 4. Do you get more useful reasoning or more noise?

4. Add a new triple about another social media actor. Re-run the question about misinformation.

5. Write one question where this simple algorithm fails. Is the problem entity linking, relation scoring, missing knowledge, or graph direction?

## 9. Switching modes

The notebook is LLM-first:

- `USE_LLM_REASONING = True`: try real LLM-guided graph reasoning.
- `PROMPT_FOR_API_KEY = False`: do not interrupt automatic runs; if no key is found, use the offline keyword fallback.
- `PROMPT_FOR_API_KEY = True`: ask students for a hidden API-key prompt when no Colab Secret is found.
- `USE_LLM_REASONING = False`: force the transparent offline keyword scorer.

The LLM does **not** answer the whole question directly. It scores candidate graph steps, and the final answer is still grounded in the visible evidence path.

## References

- Google AI Studio Gemini API key guide: https://ai.google.dev/gemini-api/docs/api-key
- Think-on-Graph paper: https://arxiv.org/abs/2307.07697
- Think-on-Graph original GitHub repository: https://github.com/GasolSun36/ToG
- Think-on-Graph moved/current GitHub repository: https://github.com/DataArcTech/ToG
- Think-on-Graph 2.0 paper: https://arxiv.org/abs/2407.10805